In [ ]:
import pandas as pd
import math
import networkx as nx
import os
import subprocess
import time
from SPARQLWrapper import SPARQLWrapper, JSON, CSV, N3, XML, TURTLE
import rdflib
import re
import IPython
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
endpoint_reactome = "http://localhost:3030/reactome"
rdfFormat = "turtle"
current_directory = os.getcwd()
BioPAX_Ontology_file_path = os.path.join(current_directory, '../', 'BioPAXData', 'biopax-level3.owl')
ReactomeBioPAX_file_path = os.path.join(current_directory, '../', 'BioPAXData', 'Homo_sapiens_v94.owl')

In [ ]:
prefixes = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX chebi: <http://purl.obolibrary.org/obo/chebi/>
PREFIX chebidb: <http://purl.obolibrary.org/obo/CHEBI_>
PREFIX chebirel: <http://purl.obolibrary.org/obo/CHEBI#>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>
PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>
PREFIX reactome: <http://www.reactome.org/biopax/40/48887#>
PREFIX abstraction:<http://abstraction/#>
"""

In [ ]:
def displaySparqlResults(results):
    '''
    Displays as HTML the result of a SPARQLWrapper query in a Jupyter notebook.
    
        Parameters:
            results (dictionnary): the result of a call to SPARQLWrapper.query().convert()
    '''
    variableNames = results['head']['vars']
    tableCode = '<table><tr><th>{}</th></tr><tr>{}</tr></table>'.format('</th><th>'.join(variableNames), '</tr><tr>'.join('<td>{}</td>'.format('</td><td>'.join([row[vName]['value'] if vName in row.keys() else "&nbsp;" for vName in variableNames]))for row in results["results"]["bindings"]))
    IPython.display.display(IPython.display.HTML(tableCode))

In [ ]:
command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', ReactomeBioPAX_file_path,
    '--file', BioPAX_Ontology_file_path,
    '/reactome']

process = subprocess.Popen(command)
time.sleep(60)

In [ ]:
def create_networkx_graph(abstraction):
    # Créer le graphe en gardant le type d'interaction comme attribut
    PathwayAbstractionGraph = nx.from_pandas_edgelist(
        abstraction,
        source="pathway1",
        target="pathway2",
        edge_attr="interaction",
        create_using=nx.MultiDiGraph()
    )
    print(f"Pathway abstraction loaded in networkx: {PathwayAbstractionGraph}")
    return PathwayAbstractionGraph


def extract_pathways(abstraction):
    return list(set(abstraction["pathway1"]).union(set(abstraction["pathway2"])))

In [ ]:
PathwayAbstraction = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94Abstractions/ReactomeHomoSapiens94_PathwayAbstraction_NextStepsAncestors.csv", sep=",", header=0)
print(PathwayAbstraction.head())

## <mark> First metric: Information content based on Resnik similarity

The information content of a pathway X (Resnik formula)

$ IC(X) = -log(\frac{|I(X)|}{|I(T)|}) $

where $I(X)$ is the number of direct and undirect descendants of pathway X and $I(T)$ is the total number of pathways

Resnik similarity between pathway X and pathway Y

$ sim(X,Y) = IC(MICA(X,Y)) $ where $MICA$ is the Most Informative Common Ancestor of pathway X and pathway Y 

In [ ]:
def  information_content_resnik(NbDescPathway , NbPathways):
    return -math.log(math.fabs(NbDescPathway)/math.fabs(NbPathways)) 

def  find_ancestors_and_descendants(G, node, interaction):
    filtered_edges = [(u, v) for u, v, attr in G .edges(data =True) if attr.get("interaction") == interaction]
    subgraph = nx.DiGraph()
    subgraph.add_edges_from(filtered_edges)
    if node  in subgraph: 
        #Since edges are child->parent, we need to reverse the graph to find descendants 
        reversed_subgraph = subgraph.reverse()
        ancestors = nx.ancestors(reversed_subgraph, node)
        descendants = nx.descendants(reversed_subgraph, node)
        return ancestors, descendants
    else:
        return set(), set()

def  find_mica(G , Pathway1 , Pathway2 ):
    ancestors1, _ = find_ancestors_and_descendants(G, Pathway1, "abstraction:IsAChildOf")
    ancestors2, _ = find_ancestors_and_descendants(G, Pathway2, "abstraction:IsAChildOf")
    common_ancestors = list(set(ancestors1) & set(ancestors2))
    if common_ancestors:
        maxIC = 0
        mica = None
        for ancestor in common_ancestors:
            _, desc = find_ancestors_and_descendants(G , ancestor, "abstraction:IsAChildOf")
            IC = information_content_resnik(len(desc), len(G .nodes()))
            if IC > maxIC:
                maxIC = IC
                mica = ancestor
        return mica
    return None 

def  create_dico_IC_pathways(abstraction):
    G = create_networkx_graph(abstraction)
    pathways = extract_pathways(abstraction)
    dicoIC = {}
    for pathway in pathways:
        _, desc = find_ancestors_and_descendants(G, pathway, "abstraction:IsAChildOf")
        if len(desc) != 0:
            IC = information_content_resnik(len(desc), len(G.nodes()))
        else:
            IC = 0
        dicoIC[pathway] = IC
    nodeTable = pd.DataFrame({"Pathway": dicoIC.keys(), "IC": dicoIC.values()})
    return dicoIC, nodeTable 

def  resnik_similarity(G , node1 , node2):
    mica = find_mica(G , node1 , node2)
    if mica:
        _, desc = find_ancestors_and_descendants(G , mica, "abstraction:IsAChildOf")
        return information_content_resnik(len(desc), len(G .nodes()))
    return 0 

def  create_dico_similarity(abstraction):
    G = create_networkx_graph(abstraction)
    dicoEdges = {}
    for idx, row in abstraction.iterrows():
        if row[1] == "abstraction:NextStepPathway":
            key = f"{row[0]} (abstraction:NextStepPathway) {row[2]}"
            sim = resnik_similarity(G, row[0], row[2])
            dicoEdges[key] = sim
    edgeTable = pd.DataFrame({"shared name": dicoEdges.keys(), "weight": dicoEdges.values()})
    return dicoEdges, edgeTable   

In [ ]:
dicoIC, nodeTable = create_dico_IC_pathways(PathwayAbstraction)
print("")
print(f"dico IC: {dicoIC}") 

dicoSim, edgeTable = create_dico_similarity(PathwayAbstraction)
print("")
print(f"dico Resnik similarity: {dicoSim}") 

In [ ]:
weighted_graph = pd.DataFrame(columns=['source', 'interaction', 'target', 'weight'])
counter_rows = 0
for item, row in PathwayAbstraction.iterrows():
    if row[1] == 'abstraction:IsAChildOf':
        weighted_graph.at[counter_rows, 'source'] = row[0]
        weighted_graph.at[counter_rows, 'interaction'] = row[1]
        weighted_graph.at[counter_rows, 'target'] = row[2]
        weighted_graph.at[counter_rows, 'weight'] = pd.NA

    if row[1] == 'abstraction:NextStepPathway':
        weighted_graph.at[counter_rows, 'source'] = row[0]
        weighted_graph.at[counter_rows, 'interaction'] = row[1]
        weighted_graph.at[counter_rows, 'target'] = row[2]
        key = f'{str(row[0])} (abstraction:NextStepPathway) {str(row[2])}'
        weight = dicoSim[key]
        weighted_graph.at[counter_rows, 'weight'] = weight
    counter_rows += 1

weighted_graph.to_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94WeightedAbstractions/ReactomeHomoSapiens94_PathwayAbstraction_NextStepsAncestors_WeightICResnik.csv", sep=",", header=True, index=False)

In [ ]:
plt.hist(dicoSim.values())

## <mark> Second metric: Similarity based on the shared physical entities between pathways 


##### The extrinsic (feature-based) information content of a pathway X

$IC(X) = -log( \frac{PE(X)}{PE(T)} )$

where $PE(X)$ is the number of physical entities annotated by pathway X and $PE(T)$ is the total number of physical entities in the graph (extrinsic IC)

##### Similarity between pathways X and Y

$sim(X,Y) = IC(MICA(X,Y))$

where $MICA(X,Y)$ is an imaginary common ancestor of pathway X and pathway Y and $PE(MICA) = PE(X) + PE(Y) - PE(X \cap Y)$


### 1.1 - Compute ($PE(X)$) for each pathway: Number of physical entities per pathway

### Count the total number of PE in the BioPAX graph ($PE(T)$)

In [ ]:
query = """ 
SELECT (COUNT(DISTINCT ?entity) AS ?nbEntity)
WHERE {
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
}
"""

# execute SPARQL query
sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query)

# display results
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
displaySparqlResults(results)

nbPe = int(results["results"]["bindings"][0]["nbEntity"]["value"])

### Count the total number of PE without Complexes in the BioPAX graph ($PE(T)$)

In [ ]:
query = """ 
SELECT (COUNT(DISTINCT ?entity) AS ?nbEntity)
WHERE {
    ?entity rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
    FILTER NOT EXISTS { ?entity rdf:type bp3:Complex . }
}
"""

# execute SPARQL query
sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query)

# display results
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
displaySparqlResults(results)

nbPeWithoutComplexes = int(results["results"]["bindings"][0]["nbEntity"]["value"])

### Count the total number of Entity References ($PE(T)$)

In [ ]:
query = """ 
SELECT (COUNT(DISTINCT ?entityRef) AS ?nbEntityRef)
WHERE {
    ?entityRef rdf:type/(rdfs:subClassOf*) bp3:EntityReference .
}
"""

# execute SPARQL query
sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query)

# display results
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
displaySparqlResults(results)

nbEr = int(results["results"]["bindings"][0]["nbEntityRef"]["value"])

### Compute Information Content with PE

In [ ]:
def create_dico_PE_per_Pathway(PEPerPathwayTable):
    dicoPEperPathway = dict()
    cleanedDicoPEPerPathway = dict()
    for item, row in PEPerPathwayTable.iterrows():
        if row[0] in dicoPEperPathway.keys():
            dicoPEperPathway[row[0]] += [row[1]]
        else:
            dicoPEperPathway[row[0]] = [row[1]]
    for key, value in dicoPEperPathway.items():
        value = [x for x in value if str(x) != 'nan']
        cleanedDicoPEPerPathway[key] = value
    return cleanedDicoPEPerPathway

def compute_similarity(Pathway1, Pathway2, dicoPEperPathway, nbPETotal):
    if Pathway1 in dicoPEperPathway.keys() and Pathway2 in dicoPEperPathway.keys():
        PE1 = dicoPEperPathway[Pathway1]
        PE2 = dicoPEperPathway[Pathway2]
    if Pathway1 not in dicoPEperPathway.keys() and Pathway2 in dicoPEperPathway.keys():
        PE1 = []
        PE2 = dicoPEperPathway[Pathway2]
    if Pathway2 not in dicoPEperPathway.keys() and Pathway1 in dicoPEperPathway.keys():
        PE1 = dicoPEperPathway[Pathway1]
        PE2 = []
    PE_MICA = (len(PE1) + len(PE2)) - len(list(set(PE1) & set(PE2)))
    if PE_MICA != 0:
        ic = -math.log(PE_MICA/nbPETotal)
    else:
        ic = 0
    return ic

In [ ]:
PEperPathway = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94UtilityFiles/ReactomeHomoSapiens94_PePerPathway.csv", sep=",", header=0)
PEperPathwayWithoutComplexes = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94UtilityFiles/ReactomeHomoSapiens94_PePerPathwayWithoutComplexes.csv", sep=",", header=0)
ERperPathway = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94UtilityFiles/ReactomeHomoSapiens94_ErPerPathway.csv", sep=",", header=0)

dicoPE = create_dico_PE_per_Pathway(PEperPathway)
dicoPEWithoutComplexes = create_dico_PE_per_Pathway(PEperPathwayWithoutComplexes)
dicoER = create_dico_PE_per_Pathway(ERperPathway)

In [ ]:
dico_next_step_pe = dict()
dico_next_step_pe_without_complexes = dict()
dico_next_step_er = dict()

distribution_weight_pe = list()
distribution_weight_pe_without_complexes = list()
distribution_weight_er = list()

for item, row in PathwayAbstraction.iterrows():
    if row[1] == "abstraction:NextStepPathway":
        Pathway1 = row[0]
        Pathway2 = row[2]
        key = f"{str(Pathway1)}absraction:NextStepPathway{str(Pathway2)}"
        weight_pe = compute_similarity(Pathway1, Pathway2, dicoPE, nbPe)
        weight_pe_without_complexes = compute_similarity(Pathway1, Pathway2, dicoPEWithoutComplexes, nbPeWithoutComplexes)
        weight_er = compute_similarity(Pathway1, Pathway2, dicoER, nbEr)
        distribution_weight_pe.append(weight_pe)
        distribution_weight_pe_without_complexes.append(weight_pe_without_complexes)
        distribution_weight_er.append(weight_er)
        
        dico_next_step_pe[key] = weight_pe
        dico_next_step_pe_without_complexes[key]= weight_pe_without_complexes
        dico_next_step_er[key] = weight_er


bins = np.linspace(0, 11, 100)

print("Weights with number of physical entities")
plt.hist(distribution_weight_pe, bins, alpha=0.5, label="Similarity with number of PE")
print("Mean:", np.mean(distribution_weight_pe), "Max:", np.max(distribution_weight_pe), "Min:", np.min(distribution_weight_pe))
print(dico_next_step_pe)
print("")

print("Weights with number of physical entities without complexes")
plt.hist(distribution_weight_pe_without_complexes, bins, alpha=0.5, label="Similarity with number of PE without complexes")
print("Mean:", np.mean(distribution_weight_pe_without_complexes), "Max:", np.max(distribution_weight_pe_without_complexes), "Min:", np.min(distribution_weight_pe_without_complexes))
print(dico_next_step_pe_without_complexes)
print("")

print("Weights with number of entity references")
plt.hist(distribution_weight_er, bins, alpha=0.5, label="Similarity with number of ER")
print("Mean:", np.mean(distribution_weight_er), "Max:", np.max(distribution_weight_er), "Min:", np.min(distribution_weight_er))
print(dico_next_step_er)

plt.legend(loc='upper right')


#### Generate weighted graph

In [ ]:
weighted_graph = pd.DataFrame(columns=['source', 'interaction', 'target', 'weight'])
counter_rows = 0
for item, row in PathwayAbstraction.iterrows():
    if row[1] == 'abstraction:IsAChildOf':
        weighted_graph.at[counter_rows, 'source'] = row[0]
        weighted_graph.at[counter_rows, 'interaction'] = row[1]
        weighted_graph.at[counter_rows, 'target'] = row[2]
        weighted_graph.at[counter_rows, 'weight'] = pd.NA

    if row[1] == 'abstraction:NextStepPathway':
        weighted_graph.at[counter_rows, 'source'] = row[0]
        weighted_graph.at[counter_rows, 'interaction'] = row[1]
        weighted_graph.at[counter_rows, 'target'] = row[2]
        key = f"{str(row[0])}absraction:NextStepPathway{str(row[2])}"
        weight = dico_next_step_er[key]
        weighted_graph.at[counter_rows, 'weight'] = weight
    counter_rows += 1

print(weighted_graph.head())

weighted_graph.to_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94WeightedAbstractions/ReactomeHomoSapiens94_PathwayAbstraction_NextStepsAncestors_WeightExtrinsicIC.csv", sep=",", header=True, index=False)

## <mark> Third metric: Mutual Information

For pathway X and pathway Y: 

$ MI(X,Y) = \frac{PE(X \cap Y)}{PE(T)} \times log( \frac { \frac{PE(X \cap Y)}{PE(T)}  } { \frac{PE(X)}{PE(T)} * \frac{PE(Y)}{PE(T)} }) $

where $PE(X)$ is the number of physical entities annotated by pathway X, $PE(Y)$ is the number of physical entities annotated by pathway Y, $PE(X \cap Y)$ the physical entities annotated by both pahtway X and pathway Y and $PE(T)$ the total number of physical entities in the graph

In [ ]:
def mutual_information(Pathway1, Pathway2, dicoPEperPathway, nbPETotal):
    PE1 = dicoPEperPathway[Pathway1]
    PE2 = dicoPEperPathway[Pathway2]
    proportionPE1 = len(PE1)/nbPETotal
    proportionPE2 = len(PE2)/nbPETotal
    intersection = len(list(set(PE1) & set(PE2)))
    if intersection != 0:
        proportion_intersection = intersection/nbPETotal
        print(proportionPE1, proportionPE2, proportion_intersection)
        MI = proportion_intersection * math.log(proportion_intersection/(proportionPE1*proportionPE2))
    else:
        MI = 0
    return MI

In [ ]:
dico_next_step = dict()
distribution_weight = list()
for item, row in PathwayAbstraction.iterrows():
    if row[1] == "abstraction:NextStepPathway":
        Pathway1 = row[0]
        Pathway2 = row[2]
        key = f"{str(Pathway1)}absraction:NextStepPathway{str(Pathway2)}"
        weight = mutual_information(Pathway1, Pathway2, dicoPE, nbPe)
        distribution_weight.append(weight)
        dico_next_step[key] = weight

print(dico_next_step)
print(distribution_weight)
plt.hist(distribution_weight)
# print(np.mean(distribution_weight))
# print(dico_next_step)

#### Generate weighted graph

In [ ]:
process.kill()
time.sleep(60)